In [1]:
# ==============================================================================
# 1. INITIALIZE ENVIRONMENT AND IMPORT CORE LIBRARIES
# ==============================================================================
import pandas as pd
import numpy as np

# Confirm environment setup
print("RetailPulse - Data Understanding")

RetailPulse - Data Understanding


In [2]:
# ==============================================================================
# 2. LOAD RAW DATASET FROM RAW DATA DIRECTORY
# ==============================================================================
# Define path to raw Excel dataset
file_path = "../data/raw/Online Retail.xlsx"

# Read Excel file into a Pandas DataFrame
df = pd.read_excel(file_path)

# Display the first 5 rows to verify initial data load
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [3]:
# ==============================================================================
# 3. INSPECT DATASET DIMENSIONS (ROWS AND COLUMNS)
# ==============================================================================
# Get total row and column count of the loaded DataFrame
df.shape

(541909, 8)

In [4]:
# ==============================================================================
# 4. INSPECT COLUMN NAMES
# ==============================================================================
# Display all column names present in the dataset
df.columns

Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country'],
      dtype='str')

In [5]:
# ==============================================================================
# 5. INSPECT DATA TYPES AND MISSING VALUE COUNTS
# ==============================================================================
# Print structural information, non-null counts, and memory usage
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[us]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  str           
dtypes: datetime64[us](1), float64(2), int64(1), object(3), str(1)
memory usage: 33.1+ MB


In [6]:
# ==============================================================================
# 6. GENERATE DESCRIPTIVE STATISTICS FOR NUMERICAL COLUMNS
# ==============================================================================
# Summarize count, mean, std, min, percentiles, and max for numeric features
df.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


## 2. Data Quality Investigation

Before performing business analysis, we systematically investigate the raw dataset 
for missing values, duplicates, cancellations, negative quantities, zero unit prices, 
and date coverage without modifying the underlying data.

In [7]:
# ==============================================================================
# 7. INVESTIGATE MISSING VALUES ACROSS ALL COLUMNS
# ==============================================================================
# Count null values per column
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [8]:
# ==============================================================================
# 8. CALCULATE MISSING VALUE PERCENTAGES
# ==============================================================================
# Compute percentage of missing data sorted from highest to lowest
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_percentage.sort_values(ascending=False)

CustomerID     24.926694
Description     0.268311
StockCode       0.000000
InvoiceNo       0.000000
Quantity        0.000000
InvoiceDate     0.000000
UnitPrice       0.000000
Country         0.000000
dtype: float64

In [9]:
# ==============================================================================
# 9. CHECK FOR DUPLICATE ROWS
# ==============================================================================
# Count exact row duplicates across the dataset
df.duplicated().sum()

np.int64(5268)

In [10]:
# ==============================================================================
# 10. DISPLAY SAMPLE DUPLICATE TRANSACTIONS
# ==============================================================================
# Preview top duplicate rows for quality assessment
df[df.duplicated()].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
555,536412,22327,ROUND SNACK BOXES SET OF 4 SKULLS,1,2010-12-01 11:49:00,2.95,17920.0,United Kingdom


In [11]:
# ==============================================================================
# 11. QUANTIFY CANCELLED TRANSACTIONS
# ==============================================================================
# Identify invoices starting with 'C' (cancellations/returns)
cancelled_orders = df["InvoiceNo"].astype(str).str.startswith("C")
cancelled_orders.sum()

np.int64(9288)

In [12]:
# ==============================================================================
# 12. CALCULATE CANCELLED TRANSACTIONS PERCENTAGE
# ==============================================================================
# Calculate proportion of cancellation records relative to total rows
(cancelled_orders.mean() * 100)

np.float64(1.7139409015166756)

In [13]:
# ==============================================================================
# 13. QUANTIFY NEGATIVE QUANTITIES
# ==============================================================================
# Count rows where item quantity is negative
negative_quantity = df["Quantity"] < 0
negative_quantity.sum()

np.int64(10624)

In [14]:
# ==============================================================================
# 14. CALCULATE NEGATIVE QUANTITY PERCENTAGE
# ==============================================================================
# Calculate percentage of transactions with negative quantities
(negative_quantity.mean() * 100)

np.float64(1.9604767590130447)

In [15]:
# ==============================================================================
# 15. INSPECT SAMPLE NEGATIVE QUANTITY TRANSACTIONS
# ==============================================================================
# Display sample records with negative quantities
df.loc[negative_quantity].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [16]:
# ==============================================================================
# 16. COUNT ZERO OR NEGATIVE UNIT PRICES
# ==============================================================================
# Count transactions with zero or negative price values
(df["UnitPrice"] <= 0).sum()

np.int64(2517)

In [17]:
# ==============================================================================
# 17. INSPECT SAMPLE ZERO OR NEGATIVE PRICE RECORDS
# ==============================================================================
# Display sample records where price is less than or equal to zero
df.loc[df["UnitPrice"] <= 0].head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom


In [18]:
# ==============================================================================
# 18. COUNT UNIQUE COUNTRIES IN DATASET
# ==============================================================================
# Get total count of distinct geographic locations
df["Country"].nunique()

38

In [19]:
# ==============================================================================
# 19. LIST ALL UNIQUE COUNTRIES
# ==============================================================================
# Display array of all unique country names
df["Country"].unique()

<StringArray>
[      'United Kingdom',               'France',            'Australia',
          'Netherlands',              'Germany',               'Norway',
                 'EIRE',          'Switzerland',                'Spain',
               'Poland',             'Portugal',                'Italy',
              'Belgium',            'Lithuania',                'Japan',
              'Iceland',      'Channel Islands',              'Denmark',
               'Cyprus',               'Sweden',              'Austria',
               'Israel',              'Finland',              'Bahrain',
               'Greece',            'Hong Kong',            'Singapore',
              'Lebanon', 'United Arab Emirates',         'Saudi Arabia',
       'Czech Republic',               'Canada',          'Unspecified',
               'Brazil',                  'USA',   'European Community',
                'Malta',                  'RSA']
Length: 38, dtype: str

In [20]:
# ==============================================================================
# 20. GEOGRAPHIC DISTRIBUTION - TOP 15 COUNTRIES
# ==============================================================================
# Count transactions per country and display top 15
df["Country"].value_counts().head(15)

Country
United Kingdom     495478
Germany              9495
France               8557
EIRE                 8196
Spain                2533
Netherlands          2371
Belgium              2069
Switzerland          2002
Portugal             1519
Australia            1259
Norway               1086
Italy                 803
Channel Islands       758
Finland               695
Cyprus                622
Name: count, dtype: int64

In [21]:
# ==============================================================================
# 21. CHECK TRANSACTION START DATE
# ==============================================================================
# Find earliest invoice timestamp in dataset
df["InvoiceDate"].min()

Timestamp('2010-12-01 08:26:00')

In [22]:
# ==============================================================================
# 22. CHECK TRANSACTION END DATE
# ==============================================================================
# Find latest invoice timestamp in dataset
df["InvoiceDate"].max()

Timestamp('2011-12-09 12:50:00')

In [23]:
# ==============================================================================
# 23. COUNT UNIQUE CUSTOMERS
# ==============================================================================
# Get total number of distinct customer IDs
df["CustomerID"].nunique()

4372

In [24]:
# ==============================================================================
# 24. COUNT UNIQUE PRODUCTS (STOCK CODES)
# ==============================================================================
# Get total number of distinct products/stock items
df["StockCode"].nunique()

4070

In [25]:
# ==============================================================================
# CALCULATE PERCENTAGE OF TRANSACTIONS MISSING CUSTOMERID
# ==============================================================================
(df["CustomerID"].isnull().mean() * 100).round(2)

np.float64(24.93)

In [26]:
# ==============================================================================
# INSPECT SAMPLE DUPLICATE TRANSACTIONS (BOTH ORIGINAL AND DUPLICATE OCCURRENCES)
# ==============================================================================
df.loc[df.duplicated(keep=False)].sort_values(by=["InvoiceNo", "StockCode"]).head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
494,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
517,536409,21866,UNION JACK FLAG LUGGAGE TAG,1,2010-12-01 11:45:00,1.25,17908.0,United Kingdom
485,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
539,536409,22111,SCOTTIE DOG HOT WATER BOTTLE,1,2010-12-01 11:45:00,4.95,17908.0,United Kingdom
489,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
527,536409,22866,HAND WARMER SCOTTY DOG DESIGN,1,2010-12-01 11:45:00,2.10,17908.0,United Kingdom
521,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
537,536409,22900,SET 2 TEA TOWELS I LOVE LONDON,1,2010-12-01 11:45:00,2.95,17908.0,United Kingdom
565,536412,21448,12 DAISY PEGS IN WOOD BOX,2,2010-12-01 11:49:00,1.65,17920.0,United Kingdom
578,536412,21448,12 DAISY PEGS IN WOOD BOX,1,2010-12-01 11:49:00,1.65,17920.0,United Kingdom


In [27]:
# ==============================================================================
# INSPECT TRANSACTIONS WITH ZERO OR NEGATIVE UNIT PRICES
# ==============================================================================
df.loc[df["UnitPrice"] <= 0, ["InvoiceNo", "StockCode", "Description", "Quantity", "InvoiceDate", "UnitPrice", "CustomerID"]].head(15)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN


In [28]:
# ==============================================================================
# COMPARE CANCELLATION INVOICES AGAINST NEGATIVE QUANTITY TRANSACTIONS
# ==============================================================================
cancellation_mask = df["InvoiceNo"].astype(str).str.startswith("C")
negative_quantity_mask = df["Quantity"] < 0

print("Cancellation invoices:", cancellation_mask.sum())
print("Negative-quantity rows:", negative_quantity_mask.sum())
print("Both conditions:", (cancellation_mask & negative_quantity_mask).sum())
print("Negative quantity without C invoice:", (~cancellation_mask & negative_quantity_mask).sum())

Cancellation invoices: 9288
Negative-quantity rows: 10624
Both conditions: 9288
Negative quantity without C invoice: 1336
